---

## Task 4: Data analysis using Pandas

### Task 4.1: Create the DataFrame

In [2]:
def create_dataframe(records: list[dict]) -> pd.DataFrame:
    """Turn the list of sales into a DataFrame sorted by time."""
    df = pd.DataFrame(records)
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df = df.set_index("timestamp")
    df = df.sort_index()
    return df


df = create_dataframe(records)
df.head()

,register_id,item,price,currency,status
timestamp,,,,,
2026-10-15 07:00:14,POS_01,Sandwich,7.55,USD,SUCCESS
2026-10-15 07:00:50,POS_01,Croissant,4.26,EUR,CANCELLED
2026-10-15 07:02:34,POS_01,Cappuccino,4.59,USD,CANCELLED
2026-10-15 07:04:07,POS_01,Espresso,2.91,NOK,CANCELLED
2026-10-15 07:04:38,POS_01,Cappuccino,4.46,EUR,REFUND


### Task 4.2: Hourly revenue per item

In [3]:
def hourly_revenue(df: pd.DataFrame) -> pd.DataFrame:
    """Total price per item for each hour."""
    result = df.groupby("item").resample("1h")["price"].sum()
    result = result.reset_index()
    return result


hourly = hourly_revenue(df)
hourly.head(10)

,item,timestamp,price
0,Cappuccino,2026-10-15 07:00:00,21.04
1,Cappuccino,2026-10-15 08:00:00,30.76
2,Cappuccino,2026-10-15 09:00:00,3.94
3,Croissant,2026-10-15 07:00:00,1033.61
4,Croissant,2026-10-15 08:00:00,11.14
5,Croissant,2026-10-15 09:00:00,7.47
6,Espresso,2026-10-15 07:00:00,12.24
7,Espresso,2026-10-15 08:00:00,24.48
8,Latte,2026-10-15 07:00:00,39.60
9,Latte,2026-10-15 08:00:00,1026.21


### Task 4.3: Pivot table of revenue by item and status

In [4]:
def status_pivot_table(df: pd.DataFrame) -> pd.DataFrame:
    """Total price per item, split into SUCCESS, REFUND and CANCELLED."""
    pivot = df.pivot_table(
        values="price",
        index="item",
        columns="status",
        aggfunc="sum",
        fill_value=0,
    )
    return pivot


pivot = status_pivot_table(df)
pivot

status,CANCELLED,REFUND,SUCCESS
item,,,
Cappuccino,4.59,22.27,28.88
Croissant,19.06,3.78,1029.38
Espresso,6.40,11.75,18.57
Latte,22.70,8.88,1039.04
Muffin,3.99,3.37,31.77
Sandwich,15.18,15.58,59.12


### Task 4.4: Refund ratio per item

We count the number of SUCCESS and REFUND transactions for each item, and
use `compute_refund_ratio` from Task 3 to calculate the ratio.

In [5]:
def refund_ratio_per_item(df: pd.DataFrame) -> pd.DataFrame:
    """Refund ratio for each item, using compute_refund_ratio from Task 3."""
    counts = pd.crosstab(df["item"], df["status"])

    sales_count = counts["SUCCESS"].to_numpy()
    refund_count = counts["REFUND"].to_numpy()
    ratio = compute_refund_ratio(sales_count, refund_count)

    result = pd.DataFrame({
        "item": counts.index,
        "sales_count": sales_count,
        "refund_count": refund_count,
        "refund_ratio": ratio,
    })
    return result


refund_ratios = refund_ratio_per_item(df)
refund_ratios

,item,sales_count,refund_count,refund_ratio
0,Cappuccino,7,5,0.714286
1,Croissant,9,1,0.111111
2,Espresso,6,4,0.666667
3,Latte,10,2,0.200000
4,Muffin,9,1,0.111111
5,Sandwich,8,2,0.250000




### Discussion of Task 4

`create_dataframe` turns the sales into a DataFrame sorted by time.
`hourly_revenue` shows the total price per item for each hour, and
`status_pivot_table` shows the total price per item for each status.
`refund_ratio_per_item` uses `compute_refund_ratio` from Task 3, with the
number of SUCCESS transactions as the sales count.

Here we only test on one log file with raw data, so the 999.00 glitches and
the wrong NOK prices are still included. These are fixed in Task 5.